## NB_group6 — Code Thuần
Implement Multinomial Naive Bayes **không dùng thư viện ML** (không sklearn, không torch).

**Thuật toán:** Maximum Likelihood Estimation với Laplace Smoothing

Pipeline:
```
Load dữ liệu → Tính priors và word counts → Tính log probabilities → Predict → Đánh giá
```

### Import (chỉ numpy + scipy)

In [17]:
import numpy as np 
import scipy.sparse
import pickle

# Không dùng sklearn hay bất kỳ ML library nào!
print(' Import xong!')

 Import xong!


### Load dữ liệu đã tiền xử lý

In [18]:
# Load ma trận TF-IDF và nhãn từ bước tiền xử lý
X_train = scipy.sparse.load_npz('../data/X_train.npz')
X_test  = scipy.sparse.load_npz('../data/X_test.npz')
y_train = np.load('../data/y_train.npy')
y_test  = np.load('../data/y_test.npy')

print(f'X_train shape : {X_train.shape}')
print(f'X_test  shape : {X_test.shape}')
print(f'y_train — ham: {int((y_train==0).sum())}  spam: {int((y_train==1).sum())}')
print(f'y_test  — ham: {int((y_test==0).sum())}  spam: {int((y_test==1).sum())}')

X_train shape : (7224, 5000)
X_test  shape : (1032, 5000)
y_train — ham: 3612  spam: 3612
y_test  — ham: 904  spam: 128


---
### Lý thuyết

### Multinomial Naive Bayes
Giả sử features (từ trong văn bản) tuân theo phân phối Multinomial.

### Priors
$$P(c) = \frac{N_c}{N}$$

### Likelihood
$$P(x_i | c) = \frac{count(x_i, c) + \alpha}{ \sum_{j} count(x_j, c) + \alpha \cdot V }$$

### Log Likelihood
$$\log P(x_i | c) = \log(count(x_i, c) + \alpha) - \log( \sum_{j} count(x_j, c) + \alpha \cdot V )$$

### Dự đoán
$$\hat{y} = \arg\max_c \left[ \log P(c) + \sum_{i} x_i \cdot \log P(x_i | c) \right]$$

### Laplace Smoothing
Thêm $\alpha$ để tránh probability = 0.

### Implement NB_group6

In [19]:
class NB_group6:
    """
    Multinomial Naive Bayes tự implement bằng Python thuần.
    Dựa trên giả định phân phối Multinomial cho features.
    """

    def __init__(self, alpha=1.0):
        """
        Parameters
        ----------
        alpha : float — smoothing parameter (Laplace smoothing)
        """
        self.alpha = alpha
        self.classes = None
        self.priors = None
        self.word_counts = None
        self.log_word_probs = None
        self.log_priors = None

    def fit(self, X, y):
        """
        Fit the model theo Multinomial Naive Bayes.
        
        Parameters
        ----------
        X : sparse matrix hoặc array, shape (n_samples, n_features)
        y : array, shape (n_samples,)
        """
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)

        self.priors = np.zeros(n_classes)
        self.word_counts = np.zeros((n_classes, n_features))

        for idx, c in enumerate(self.classes):
            mask = (y == c)
            X_c = X[mask]
            self.priors[idx] = X_c.shape[0] / n_samples
            self.word_counts[idx] = np.array(X_c.sum(axis=0)).flatten()

        total_count_per_class = self.word_counts.sum(axis=1)
        self.log_word_probs = np.log((self.word_counts + self.alpha) / 
                                    (total_count_per_class[:, None] + self.alpha * n_features))
        self.log_priors = np.log(self.priors)

    def predict_proba(self, X):
        """
        Dự đoán xác suất cho mỗi class.
        
        Parameters
        ----------
        X : sparse matrix hoặc array, shape (n_samples, n_features)
        
        Returns
        -------
        proba : array, shape (n_samples, n_classes)
        """
        log_scores = X.dot(self.log_word_probs.T) + self.log_priors
        # Normalize để thành xác suất
        log_scores -= np.max(log_scores, axis=1, keepdims=True)  # tránh overflow
        scores = np.exp(log_scores)
        return scores / scores.sum(axis=1, keepdims=True)

    def predict(self, X):
        """
        Dự đoán class với xác suất cao nhất.
        
        Parameters
        ----------
        X : sparse matrix hoặc array, shape (n_samples, n_features)
        
        Returns
        -------
        y_pred : array, shape (n_samples,)
        """
        scores = X.dot(self.log_word_probs.T) + self.log_priors
        return self.classes[np.argmax(scores, axis=1)]

    def score(self, X, y):
        """
        Tính accuracy trên tập dữ liệu.
        
        Parameters
        ----------
        X : sparse matrix hoặc array, shape (n_samples, n_features)
        y : array, shape (n_samples,)
        
        Returns
        -------
        accuracy : float
        """
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

### Train model

### Evaluate

In [20]:
# Train model
model = NB_group6(alpha=1.0)
model.fit(X_train, y_train)

In [21]:
# Evaluate
train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)

print(f"Train Accuracy: {train_acc*100:.2f}%")
print(f"Test Accuracy: {test_acc*100:.2f}%")

Train Accuracy: 98.41%
Test Accuracy: 95.45%


### Phân tích từ khóa

In [22]:
# Load vectorizer
with open('../data/tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

words = np.array(vectorizer.get_feature_names_out())

# Find spam class index safely
spam_idx = np.where(model.classes == 1)[0][0]

top_10_spam_indices = np.argsort(model.log_word_probs[spam_idx])[-10:][::-1]

print("Top 10 Spam words:")
for i, idx in enumerate(top_10_spam_indices):
    print(f"{i+1}. {words[idx]}")

Top 10 Spam words:
1. call
2. free
3. txt
4. claim
5. mobile
6. prize
7. stop
8. text
9. reply
10. urgent
